# Automação de Busca de Vagas - Gupy

Projeto de automação com Selenium para busca e coleta de vagas no Gupy.

## 1. Configuração inicial

Importação de bibliotecas e configuração do navegador.

In [ ]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time
import pandas as pd


## 2. Busca automatizada

Abrir o site, preencher a busca e confirmar.

In [2]:
# criar o navegador
servico = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=servico)
driver.get("https://portal.gupy.io/job-search")
driver.maximize_window()
time.sleep(5)


In [3]:
driver.find_element(By.CSS_SELECTOR, 'input[placeholder="Busque por uma vaga"]').send_keys("Estágio TI")
driver.find_element(By.CSS_SELECTOR, 'button[data-testid="search-button"]').click()


In [4]:
driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Local de trabalho"]').click()
time.sleep(1)

driver.find_element(By.ID, "dropdown-location-state-input").send_keys("São Paulo")
time.sleep(3)

opcoes_estado = driver.find_elements(By.CSS_SELECTOR, '[role="option"]')

estado = "São Paulo (SP)"

for opcao in opcoes_estado:
    if opcao.text == estado:
        opcao.click()
        break

time.sleep(3)

driver.find_element(By.ID, "multi-value-dropdown-location-city-input").send_keys("São Paulo")
time.sleep(2)

opcoes_cidades = driver.find_elements(By.CSS_SELECTOR, '[role="option"]')

cidade = "São Paulo"

for opcao in opcoes_cidades:
    if opcao.text == cidade:
        opcao.click()
        break

time.sleep(2)

driver.find_element(By.ID, "multi-value-dropdown-location-city-input").click()

time.sleep(3)
driver.find_element(By.XPATH, '//button[normalize-space()="Aplicar"]').click()




In [5]:
'''driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Modelo de trabalho"]').click()

driver.find_element(By.NAME, "on-site").click()

driver.find_element(By.NAME, "hybrid").click()

driver.find_element(By.NAME, "remote").click()

driver.find_element(By.XPATH, '//button[normalize-space()="Aplicar"]').click()'''

'driver.find_element(By.CSS_SELECTOR, \'button[aria-label="Modelo de trabalho"]\').click()\n\ndriver.find_element(By.NAME, "on-site").click()\n\ndriver.find_element(By.NAME, "hybrid").click()\n\ndriver.find_element(By.NAME, "remote").click()\n\ndriver.find_element(By.XPATH, \'//button[normalize-space()="Aplicar"]\').click()'

## 3. Extração dos dados da página de resultados

Identificar e extrair título, empresa, local e link de cada vaga.

In [21]:
lista_vagas = []

vagas = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/job/"]')

for vaga in vagas:
    
    titulo = vaga.find_element(By.TAG_NAME, "h3").text

    empresas = vaga.find_elements(By.TAG_NAME, "p")

    for empresa in empresas:
        if not empresa.text.startswith("Publicada em:"):
            nome_empresa = empresa.text
        else:
            data_vaga_publicada = empresa.text

    local = vaga.find_elements(By.CSS_SELECTOR, 'span[data-testid="job-location"]')

    if len(local) == 0:
        local_vaga = None
    else:
        local_vaga = local[0].text

    modelos_trabalho = ["Presencial", "Híbrido", "Remoto"]

    tipos_vaga = ["Estágio", "Efetivo", "Associado", "Autônomo", "Temporário", "Pessoa Jurídica", "Trainee", "Sócio"]

    elementos_span = vaga.find_elements(By.TAG_NAME, "span")

    modelo_encontrado = None
    tipo_vaga_encontrada = None
    pcd_encontrado = None

    for el_span in elementos_span:
        if el_span.text in modelos_trabalho:
            modelo_encontrado = el_span.text

        elif el_span.text in tipos_vaga:
            tipo_vaga_encontrada = el_span.text

        elif el_span.text == "Também p/ PcD":
            pcd_encontrado = el_span.text
            
    link = vaga.get_attribute("href")


    dic_vagas = {
        "Titulo": titulo,
        "Empresa": nome_empresa,
        "Local": local_vaga,
        "Modelo": modelo_encontrado,
        "Tipo da Vaga": tipo_vaga_encontrada,
        "Afirmativa para PcD": pcd_encontrado,
        "Data": data_vaga_publicada,
        "Link": link
        }

    lista_vagas.append(dic_vagas)
    


## 4. Paginação

Navegar pelas próximas páginas de resultados, se necessário.

## 5. Organização e exportação dos dados

Transformar os resultados em DataFrame e exportar para CSV.

In [61]:
tabela_vagas = pd.DataFrame(lista_vagas)

tabela_vagas["Data"] = tabela_vagas["Data"].str.replace("Publicada em:", "", regex=False).str.strip()

tabela_vagas["Data"] = pd.to_datetime(tabela_vagas["Data"], format='%d/%m/%Y')

colunas_nulos = [col for col in tabela_vagas.columns if col != "Data"]

for coluna in colunas_nulos:
    tabela_vagas[coluna] = tabela_vagas[coluna].fillna("Não informado") 

tabela_vagas.to_csv("reports/lista_vagas.csv", index=False, sep=";", encoding="utf-8-sig", date_format='%d/%m/%Y')

display(tabela_vagas)

,Titulo,Empresa,Local,Modelo,Tipo da Vaga,Afirmativa para PcD,Data,Link
0,Estágio em TI | São Paulo/SP,Geração Cyrela,São Paulo - SP,Híbrido,Estágio,Também p/ PcD,2026-08-10,https://geracaocyrela.gupy.io/job/eyJqb2JJZCI6...
1,Estágio Universitário - Suporte TI,Trabalhe na Mills,São Paulo - SP,Presencial,Estágio,Também p/ PcD,2026-08-05,https://programadeestagiomills.gupy.io/job/eyJ...
2,Estágio Nível Superior - TI - Segurança da Inf...,CSN - Companhia Siderúrgica Nacional,São Paulo - SP,Presencial,Estágio,Também p/ PcD,2026-07-20,https://csn.gupy.io/job/eyJqb2JJZCI6MTE2NDkzND...
3,Estágio em Suporte de TI - HÍBRIDO,#sejaveriter,São Paulo - SP,Híbrido,Efetivo,Também p/ PcD,2026-06-29,https://verity.gupy.io/job/eyJqb2JJZCI6MTE1NTg...
4,Estágio em TI,BANCO DAYCOVAL,São Paulo - SP,Presencial,Efetivo,Também p/ PcD,2026-06-12,https://bancodaycoval.gupy.io/job/eyJqb2JJZCI6...
5,Banco de Talentos Estágio em Desenvolvimento - TI,Gertec Brasil,Não informado,Não informado,Não informado,Não informado,2024-02-15,https://gertec.gupy.io/job/eyJqb2JJZCI6NjcyMzc...
